# 💼 Multi-Feature Tech Salary Predictor (India - ₹ LPA)

This notebook demonstrates an end-to-end Machine Learning pipeline for predicting tech compensation (in Lakhs Per Annum - ₹ LPA) based on experience, role, education, location, and company scale.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

## 1. Load Dataset & Exploratory Data Analysis

In [ ]:
df = pd.read_csv('data/salary_data_multifeature.csv')
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nSummary Statistics (Salaries in ₹ LPA):")
df.describe()

## 2. Visualizing Compensation Trends

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='years_of_experience', y='salary_in_lpa', hue='job_title', style='education_level', s=100)
plt.title('Experience vs Salary (₹ LPA)')
plt.xlabel('Years of Experience')
plt.ylabel('Salary (₹ LPA)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. Preprocessing & Model Benchmarking

In [ ]:
X = df.drop(columns=['salary_in_lpa'])
y = df['salary_in_lpa']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['years_of_experience']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['job_title', 'education_level', 'location', 'company_size'])
    ]
)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append({
        'Model': name,
        'R2 Score': r2_score(y_test, preds),
        'MAE (₹ LPA)': mean_absolute_error(y_test, preds),
        'RMSE (₹ LPA)': np.sqrt(mean_squared_error(y_test, preds))
    })

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
results_df